### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output. 

### Pydantic

Pydantic models provide the richest feature set with field validation, description and nested structure.

In [4]:
from dotenv import load_dotenv
load_dotenv()
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024A6D6AE270>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024A6D6AEF90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [6]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024A6D6AE270>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024A6D6AEF90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out 

In [7]:
response=model_with_structure.invoke("Provide details about the movie inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [8]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movie rating out of 10")
    
model_with_structure=model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("Provide details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception". Let me see what I need to do here. The available tool is the Movie function, which requires title, year, director, and rating. I need to make sure I have all these parameters.\n\nFirst, the title is "Inception". The year it was released is 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb, but since the user didn\'t specify a source, I\'ll just use a general rating. Let me confirm these details. Yeah, "Inception" came out in 2010, directed by Christopher Nolan, and it has an 8.8/10 rating. \n\nSo I need to structure the function call with these parameters. The required fields are title, year, director, and rating. All are present here. Let me format the JSON correctly. Make sure the types are right: title and director are strings, year is integer, rating is a number. No typos in the keys. Alright, that should

### Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
    
class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million USD")
    
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [10]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year movie was released"]
    director: Annotated[str, ..., "the director of the movie"]
    rating: Annotated[float, ..., "the movie rating out of 10"]
    
model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie avenger")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [11]:
class Actor(TypedDict):
    name:str
    role:str
    
class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million USD")
    
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie avenger endgame")
response

{'budget': 356000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'}],
 'genres': ['Action', 'Science Fiction', 'Superhero'],
 'title': 'Avenger Endgame',
 'year': 2019}

In [12]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [13]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [ ]:
from pydantic import BaseModel , Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="the name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")
    
agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user","content": "extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='john Doe', email='john@example.com', phone='(555) 123-4567)

result

In [ ]:
result["structured_response"]

In [ ]:
### Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person """
    name: str = Field(description="the name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")
    
agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user","content": "extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["5-structured_response"]

In [ ]:
## Data class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person """
    name: str = Field(description="the name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")
    
agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user","content": "extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["5-structured_response"]
